In [1]:
!pip install --upgrade protobuf
import tensorflow as tf
from tensorflow.keras.datasets import imdb

# Load the IMDB dataset (keeping the top 10,000 most frequent words)
(train_data, train_labels), (test_data, test_labels) = imdb.load_data(num_words=10000)

# Example: Decoding the first review back to text
word_index = imdb.get_word_index()
reverse_word_index = dict([(value, key) for (key, value) in word_index.items()])
decoded_review = ' '.join([reverse_word_index.get(i - 3, '?') for i in train_data[0]])

print(decoded_review)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
? this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert ? is an amazing actor and now the same being director ? father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for ? and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also ? to the two little boy's that played the ? of norman and paul they were just brilliant children are often left out of the ? list i think because the stars that play them all grown up are such a big profile for the w

In [5]:
import pandas as pd
import numpy as np

# Function to decode a numerical review back into text
def decode_review(text_numbers, reverse_word_index):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in text_numbers])

# Combine train and test data and labels
all_data = np.concatenate((train_data, test_data), axis=0)
all_labels = np.concatenate((train_labels, test_labels), axis=0)

# Decode all reviews
decoded_reviews = [decode_review(review, reverse_word_index) for review in all_data]

# Create a single combined DataFrame
combined_df = pd.DataFrame({
    'text': decoded_reviews,
    'label': all_labels
})

print("Combined DataFrame head:")
display(combined_df.head())
print(f"Combined DataFrame shape: {combined_df.shape}")
df = combined_df

Combined DataFrame head:


,text,label
0,? this film was just brilliant casting locatio...,1
1,? big hair big boobs bad music and a giant saf...,0
2,? this has to be one of the worst films of the...,0
3,? the ? ? at storytelling the traditional sort...,1
4,? worst mistake of my life br br i picked this...,0


Combined DataFrame shape: (50000, 2)


# Preprocessing


In [30]:
import string
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [31]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))


In [33]:
from nltk.tokenize import word_tokenize

# Required model resource download
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [32]:
#removing punctuation and stopwords
def clean_words(words):
  cleaned_words = [
          lemmatizer.lemmatize(word.lower()) for word in words
          if word not in stop_words and word not in string.punctuation
      ]
  return cleaned_words


In [35]:
# The 'text' column of the DataFrame 'df' already contains lists of words,
# as shown in the kernel state (e.g., ['film', 'brilliant', ...]).
# Therefore, `word_tokenize` is not needed, as the text is already tokenized.
# The error `TypeError: expected string or bytes-like object, got 'list'` occurs
# because `word_tokenize` is being called with a list (`text_list[i]`) instead of a string.

# Additionally, the loop structure was incorrectly reassigning `text_list` in each iteration.
# We can use the pandas `.apply()` method to correctly apply the cleaning function.

text = df['text'][:1000].apply(clean_words)

# If you want to convert the lists of cleaned words back into single strings, uncomment the next line:
# df['text'] = df['text'].apply(lambda x: ' '.join(x))

print("DataFrame after cleaning words:")
display(text)

DataFrame after cleaning words:


,text
0,"[film, brilliant, casting, location, scenery, ..."
1,"[big, hair, big, boob, bad, music, giant, safe..."
2,"[one, worst, film, 1990s, friend, watching, fi..."
3,"[storytelling, traditional, sort, many, year, ..."
4,"[worst, mistake, life, br, br, picked, movie, ..."
...,...
995,"[think, still, best, routine, others, like, br..."
996,"[thirteen, year, old, saw, movie, expected, lo..."
997,"[record, production, way, br, br, hidden, fron..."
998,"[super, sexy, b, movie, actress, another, bit,..."


In [46]:
x = text.to_numpy()
y = df['label'][:1000]

In [48]:
print("X :",x)
print("Y :",y)

X : [list(['film', 'brilliant', 'casting', 'location', 'scenery', 'story', 'direction', 'everyone', "'s", 'really', 'suited', 'part', 'played', 'could', 'imagine', 'robert', 'amazing', 'actor', 'director', 'father', 'came', 'scottish', 'island', 'loved', 'fact', 'real', 'connection', 'film', 'witty', 'remark', 'throughout', 'film', 'great', 'brilliant', 'much', 'bought', 'film', 'soon', 'released', 'would', 'recommend', 'everyone', 'watch', 'fly', 'fishing', 'amazing', 'really', 'cried', 'end', 'sad', 'know', 'say', 'cry', 'film', 'must', 'good', 'definitely', 'also', 'two', 'little', 'boy', "'s", 'played', 'norman', 'paul', 'brilliant', 'child', 'often', 'left', 'list', 'think', 'star', 'play', 'grown', 'big', 'profile', 'whole', 'film', 'child', 'amazing', 'praised', 'done', "n't", 'think', 'whole', 'story', 'lovely', 'true', 'someone', "'s", 'life', 'shared', 'u'])
 list(['big', 'hair', 'big', 'boob', 'bad', 'music', 'giant', 'safety', 'pin', 'word', 'best', 'describe', 'terrible', 

# OHE


In [49]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import numpy as np


In [51]:
# Flatten the list of lists of words to get all words from 'text'
# 'text' is df['text'][:1000]
all_words_from_text = [word for sublist in text for word in sublist]

# Get unique words to build the vocabulary
unique_words = sorted(list(set(all_words_from_text)))

print(f"Total number of words extracted: {len(all_words_from_text)}")
print(f"Vocabulary size (number of unique words): {len(unique_words)}")

# Initialize LabelEncoder to convert words to integers
label_encoder = LabelEncoder()

# Fit and transform unique words to get integer encodings for the vocabulary
integer_encoded_vocab = label_encoder.fit_transform(unique_words)

# Reshape for OneHotEncoder (requires 2D input)
integer_encoded_vocab_reshaped = integer_encoded_vocab.reshape(len(integer_encoded_vocab), 1)

# Initialize OneHotEncoder
# handle_unknown='ignore' allows encoding words not seen during fit (they will be encoded as all zeros)
# sparse_output=True is good for memory efficiency with large vocabularies
onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)

# Fit and transform the integer encoded vocabulary to get one-hot representations
onehot_encoded_vocab = onehot_encoder.fit_transform(integer_encoded_vocab_reshaped)

print("\nSample of unique words and their integer encodings:")
# Display first few unique words and their integer encodings
for i in range(min(10, len(unique_words))):
    print(f"Word: '{unique_words[i]}', Integer ID: {integer_encoded_vocab[i]}")

print("\nShape of one-hot encoded vocabulary matrix (num_unique_words, num_unique_words):")
print(onehot_encoded_vocab.shape)
# To view a sample one-hot vector, you would convert a sparse matrix row to a dense array:
print("Sample one-hot encoded vector for first word in vocabulary:\n", onehot_encoded_vocab[0].toarray())

Total number of words extracted: 118173
Vocabulary size (number of unique words): 7692

Sample of unique words and their integer encodings:
Word: ''70s', Integer ID: 0
Word: ''73', Integer ID: 1
Word: ''80s', Integer ID: 2
Word: ''cause', Integer ID: 3
Word: ''d', Integer ID: 4
Word: ''em', Integer ID: 5
Word: ''ll', Integer ID: 6
Word: ''m', Integer ID: 7
Word: ''re', Integer ID: 8
Word: ''s', Integer ID: 9

Shape of one-hot encoded vocabulary matrix (num_unique_words, num_unique_words):
(7692, 7692)
Sample one-hot encoded vector for first word in vocabulary:
 [[1. 0. 0. ... 0. 0. 0.]]


In [57]:
from scipy.sparse import vstack, csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

def reviews_to_ohe_matrix(reviews_list, label_encoder, onehot_encoder):
    """Converts a list of word-lists into a document-term matrix using the OHE encoders."""
    encoded_reviews = []
    for words in reviews_list:
        if not words:
            # Handle empty reviews with a zero vector of the vocabulary size
            encoded_reviews.append(csr_matrix((1, len(label_encoder.classes_))))
            continue

        # Convert words to integer IDs
        word_ids = label_encoder.transform(words).reshape(-1, 1)
        # Transform to one-hot vectors and sum them to get the document vector
        doc_vector = onehot_encoder.transform(word_ids).sum(axis=0)
        encoded_reviews.append(csr_matrix(doc_vector))

    return vstack(encoded_reviews)

# Transform the data using the OHE logic
print("Vectorizing text using One-Hot Encoding...")
X_ohe = reviews_to_ohe_matrix(x, label_encoder, onehot_encoder)

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_ohe, y, test_size=0.2, random_state=42)

# Train Logistic Regression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Predictions and Evaluation
y_pred = model.predict(X_test)
print(f"\nModel Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Vectorizing text using One-Hot Encoding...

Model Accuracy: 0.7750

Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.75      0.78       106
           1       0.74      0.80      0.77        94

    accuracy                           0.78       200
   macro avg       0.78      0.78      0.77       200
weighted avg       0.78      0.78      0.78       200



### Simplified Vectorization with CountVectorizer
Instead of manually using `LabelEncoder` and `OneHotEncoder` and then summing them, `CountVectorizer` with `binary=True` performs the exact same task more efficiently.

In [58]:
from sklearn.feature_extraction.text import CountVectorizer

# Join the list of words back into strings for CountVectorizer
x_strings = [' '.join(words) for words in x]

# Initialize and fit CountVectorizer
# binary=True ensures it acts like a multi-hot/OHE document vector
vectorizer = CountVectorizer(binary=True)
X_ohe_simple = vectorizer.fit_transform(x_strings)

# Split and train as before
X_train, X_test, y_train, y_test = train_test_split(X_ohe_simple, y, test_size=0.2, random_state=42)
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)

print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"Simplified Model Accuracy: {model.score(X_test, y_test):.4f}")

Vocabulary size: 7653
Simplified Model Accuracy: 0.8000


### TF-IDF Vectorization
TF-IDF reflects how important a word is to a document in a collection. It helps the model focus on more meaningful words.

In [59]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(x_strings)

# Split and train
X_train_tf, X_test_tf, y_train_tf, y_test_tf = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)
model_tfidf = LogisticRegression(max_iter=1000).fit(X_train_tf, y_train_tf)

print(f"TF-IDF Model Accuracy: {model_tfidf.score(X_test_tf, y_test_tf):.4f}")
print(f"Vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")

TF-IDF Model Accuracy: 0.7800
Vocabulary size: 7653


### Standard Bag-of-Words (BoW)
This model uses the raw frequency counts of each word as features.

In [60]:
from sklearn.feature_extraction.text import CountVectorizer

# Initialize CountVectorizer with binary=False (default) for word counts
bow_vectorizer = CountVectorizer(binary=False)
X_bow = bow_vectorizer.fit_transform(x_strings)

# Split and train
X_train_bow, X_test_bow, y_train_bow, y_test_bow = train_test_split(X_bow, y, test_size=0.2, random_state=42)
model_bow = LogisticRegression(max_iter=1000).fit(X_train_bow, y_train_bow)

print(f"Standard BoW Model Accuracy: {model_bow.score(X_test_bow, y_test_bow):.4f}")
print(f"Example counts for first review: {X_bow[0].data[:10]}")

Standard BoW Model Accuracy: 0.7550
Example counts for first review: [6 3 1 1 1 2 1 2 2 1]


In [63]:
pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 19.3 MB/s eta 0:00:00


In [64]:

from gensim.models import Word2Vec
import numpy as np

# Function to average word vectors for a document
def get_avg_word2vec(tokens, model, vector_size):
    valid_vectors = [model.wv[word] for word in tokens if word in model.wv]
    if not valid_vectors:
        return np.zeros(vector_size)
    return np.mean(valid_vectors, axis=0)

# 1. Train CBOW Model (sg=0)
cbow_model = Word2Vec(sentences=x, vector_size=100, window=5, min_count=1, sg=0)

# 2. Train Skip-gram Model (sg=1)
skipgram_model = Word2Vec(sentences=x, vector_size=100, window=5, min_count=1, sg=1)

# Create features by averaging vectors
X_cbow = np.array([get_avg_word2vec(tokens, cbow_model, 100) for tokens in x])
X_skipgram = np.array([get_avg_word2vec(tokens, skipgram_model, 100) for tokens in x])

In [66]:
# Evaluate CBOW
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_cbow, y, test_size=0.2, random_state=42)
model_cbow = LogisticRegression(max_iter=1000).fit(X_train_c, y_train_c)
print(f"Avg CBOW Word2Vec Accuracy: {model_cbow.score(X_test_c, y_test_c):.4f}")

# Evaluate Skip-gram
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_skipgram, y, test_size=0.2, random_state=42)
model_sg = LogisticRegression(max_iter=1000).fit(X_train_s, y_train_s)
print(f"Avg Skip-gram Word2Vec Accuracy: {model_sg.score(X_test_s, y_test_s):.4f}")

Avg CBOW Word2Vec Accuracy: 0.5050
Avg Skip-gram Word2Vec Accuracy: 0.6000


### Google Universal Sentence Encoder (USE)
The Universal Sentence Encoder (USE) from Google Research encodes text into high-dimensional vectors that can be used for text classification, semantic similarity, and other natural language tasks. Unlike Word2Vec, it handles the context of the entire sentence.

In [67]:
import tensorflow_hub as hub

# 1. Load the Universal Sentence Encoder model
# This might take a moment to download the first time
use_model = hub.load("https://tfhub.dev/google/universal-sentence-encoder/4")

# 2. Convert text to embeddings
# USE expects a list of strings and returns 512-dimensional vectors
X_google_use = use_model(x_strings).numpy()

# 3. Split and Evaluate using Logistic Regression
X_train_u, X_test_u, y_train_u, y_test_u = train_test_split(X_google_use, y, test_size=0.2, random_state=42)
model_use = LogisticRegression(max_iter=1000).fit(X_train_u, y_train_u)

print(f"USE Vector shape: {X_google_use.shape}")
print(f"Google USE Accuracy: {model_use.score(X_test_u, y_test_u):.4f}")

USE Vector shape: (1000, 512)
Google USE Accuracy: 0.7650
